# 02 — The GPT-2 forward pass, reimplemented

**One tangible win:** rebuild GPT-2-small's forward pass from torch tensor ops (matmul,
softmax, mean/var — the linear-algebra layer; no `nn.Module`, no autodiff) and pin it against
the pretrained `openai-community/gpt2` with `assert reimpl_logits ≈ hf_logits`.

The shape of it: `wte[tok] + wpe[pos]` → **12 pre-LN blocks** (`x + Attn(ln_1(x))`, then
`x + MLP(ln_2(x))`) → `ln_f` → tied unembed (`@ wte.T`). HF is the reference oracle; the assert
is a *correctness check on the substrate*, not evidence of understanding.

In [ ]:
import math
import torch
import transformers

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
model = transformers.AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
device

## The reference oracle

One forward pass through HF in `no_grad` + `eval` (dropout off → deterministic). These logits —
shape `(batch, seq, vocab=50257)` — are what the reimplementation has to reproduce.

In [ ]:
prompt = "The quick brown fox"
input_ids = tokenizer(prompt, return_tensors="pt").to(device)["input_ids"]

with torch.no_grad():
    hf_logits = model(input_ids).logits

input_ids.shape, hf_logits.shape

## The weights, by name

Pull the pretrained parameters straight out of the `state_dict` — the reimpl is *only* the
arithmetic that strings them together. Config constants: 12 layers, 12 heads, d_model 768, so
head_dim = 64.

In [ ]:
sd = model.state_dict()
N_LAYER, N_HEAD, D = 12, 12, 768
HEAD_DIM = D // N_HEAD  # 64

# Spot-check the layout the matmuls depend on: Conv1D weight is (nx, nf), NOT (nf, nx).
assert sd["transformer.h.0.attn.c_attn.weight"].shape == (D, 3 * D)   # fused Q,K,V
assert sd["transformer.wte.weight"].shape == (50257, D)
sd["transformer.h.0.attn.c_attn.weight"].shape

## Primitives

- **`conv1d`** — every GPT-2 projection is a `Conv1D`: a Linear with *transposed* weights, so the
  forward is `x @ W + b` with `W` shape `(in, out)`. Get this backwards and the checkpoint
  transpose-mismatches.
- **`layer_norm`** — per-token normalize over the 768 features with **biased** variance (÷H),
  `eps=1e-5` *inside* the sqrt, then per-feature affine `* w + b`.
- **`gelu_new`** — the **tanh** approximation (HF `"gelu_new"`), not the erf form; the difference
  shows up at a tight tolerance.

In [ ]:
def conv1d(x, w, b):
    # Conv1D == transposed-weight Linear: w is (in, out), so this is x @ W + b.
    return x @ w + b

def layer_norm(x, w, b, eps=1e-5):
    mu = x.mean(-1, keepdim=True)
    var = x.var(-1, keepdim=True, unbiased=False)  # biased: divide by H, not H-1
    return (x - mu) / torch.sqrt(var + eps) * w + b

def gelu_new(x):
    return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * x ** 3)))

## Attention

Causal multi-head self-attention. The fused `c_attn` projects to `3 × 768`; `.split(768, dim=2)`
peels off Q, K, V. Reshape each to `(b, n_head, seq, head_dim)`, score with `QKᵀ` scaled by
`head_dim**-0.5` (= 1/8), apply the **causal mask** (`tril`; illegal positions → −∞ *before*
softmax — equivalent to HF's additive mask), weight V, then concat heads and project out with
`c_proj`.

In [ ]:
def attention(x, c_attn_w, c_attn_b, c_proj_w, c_proj_b):
    b, seq, d = x.shape
    q, k, v = conv1d(x, c_attn_w, c_attn_b).split(D, dim=2)

    # (b, seq, d) -> (b, n_head, seq, head_dim)
    split_heads = lambda t: t.view(b, seq, N_HEAD, HEAD_DIM).transpose(1, 2)
    q, k, v = split_heads(q), split_heads(k), split_heads(v)

    scores = (q @ k.transpose(-1, -2)) * (HEAD_DIM ** -0.5)
    causal = torch.tril(torch.ones(seq, seq, device=x.device, dtype=torch.bool))
    scores = scores.masked_fill(~causal, float("-inf"))
    out = torch.softmax(scores, dim=-1) @ v          # (b, n_head, seq, head_dim)

    out = out.transpose(1, 2).contiguous().view(b, seq, d)  # concat heads
    return conv1d(out, c_proj_w, c_proj_b)

## MLP and the full forward

The MLP is the 4× position-wise FFN: `c_fc` (768→3072) → GELU → `c_proj` (3072→768). The block
wires both sub-layers pre-LN and *adds* back into the residual stream. The whole model is embed →
12 blocks → `ln_f` → unembed, where the unembed **reuses `wte`** (weight tying) — one matrix
bookends the stream.

In [ ]:
def mlp(x, c_fc_w, c_fc_b, c_proj_w, c_proj_b):
    return conv1d(gelu_new(conv1d(x, c_fc_w, c_fc_b)), c_proj_w, c_proj_b)

@torch.no_grad()
def gpt2_forward(input_ids):
    seq = input_ids.shape[1]
    wte = sd["transformer.wte.weight"]
    pos = torch.arange(seq, device=input_ids.device)
    x = wte[input_ids] + sd["transformer.wpe.weight"][pos]   # into the residual stream

    for i in range(N_LAYER):
        p = f"transformer.h.{i}."
        x = x + attention(
            layer_norm(x, sd[p + "ln_1.weight"], sd[p + "ln_1.bias"]),
            sd[p + "attn.c_attn.weight"], sd[p + "attn.c_attn.bias"],
            sd[p + "attn.c_proj.weight"], sd[p + "attn.c_proj.bias"],
        )
        x = x + mlp(
            layer_norm(x, sd[p + "ln_2.weight"], sd[p + "ln_2.bias"]),
            sd[p + "mlp.c_fc.weight"], sd[p + "mlp.c_fc.bias"],
            sd[p + "mlp.c_proj.weight"], sd[p + "mlp.c_proj.bias"],
        )

    x = layer_norm(x, sd["transformer.ln_f.weight"], sd["transformer.ln_f.bias"])
    return x @ wte.T   # tied unembed

## The pin

`max |reimpl − hf|` should sit at float32 matmul noise (~1e-5). The `allclose` at `atol=1e-4` is
the substrate-correctness check — green means the matmuls and the layout are right. It does **not**
mean the mental model is right; that's what the interrogation is for.

In [ ]:
reimpl_logits = gpt2_forward(input_ids)

max_abs_diff = (reimpl_logits - hf_logits).abs().max().item()
print(f"shapes: {tuple(reimpl_logits.shape)} vs {tuple(hf_logits.shape)}")
print(f"max abs diff: {max_abs_diff:.2e}")
print(f"argmax (next-token) matches: {torch.equal(reimpl_logits.argmax(-1), hf_logits.argmax(-1))}")

assert torch.allclose(reimpl_logits, hf_logits, atol=1e-4), max_abs_diff
print("\nassert reimpl_logits ≈ hf_logits  ✓")